# Mistral Large 4: sparsity math, pricing math, and the refusal-vs-capability score
Companion notebook for the 2026-10-14 post. All vendor numbers are Mistral's Oct 6, 2026
selections; independent numbers from Artificial Analysis. Run top to bottom.

## 1. MoE sparsity: what 52B awake of 1.05T actually buys you

In [ ]:
total_b, active_b = 1050.0, 52.0          # billions of params
activation_ratio = active_b / total_b
print(f"Activation ratio: {100*activation_ratio:.2f}%  ({active_b:.0f}/{total_b:.0f}B)")

# Memory to HOST the weights: 1 byte/param @8-bit, 0.5 byte/param @4-bit
mem_8bit_gb = total_b * 1.0
mem_4bit_gb = total_b * 0.5
print(f"Host memory: {mem_8bit_gb:.0f} GB @8-bit, {mem_4bit_gb:.0f} GB @4-bit")
assert abs(activation_ratio - 0.0495) < 0.001
assert mem_8bit_gb == 1050.0 and mem_4bit_gb == 525.0
print("Memory sets the host bill; active params set the compute bill.")

## 2. Worked cost example: one agent session
2M input tokens (1.6M cached, 0.4M uncached) + 100K output tokens, standard tier. Rates from Mistral's pricing page (Oct 2026).

In [ ]:
def session_cost(inp, cache_inp, outp, rates):
    uncached, cached, out = rates
    return uncached*inp/1e6 + cached*cache_inp/1e6 + out*outp/1e6

cases = {
    "ML4 sale":        (0.68, 0.07, 2.09),
    "ML4 list":        (1.36, 0.14, 4.18),
    "GLM-5.3 (Mistral host)": (1.40, 0.14, 4.40),
    "Kimi K3 (Moonshot)":      (3.00, 0.30, 15.00),
}
inp, cached, outp = 0.4e6, 1.6e6, 0.1e6
for name, rates in cases.items():
    print(f"{name:24s} ${session_cost(inp, cached, outp, rates):.2f}")

assert abs(session_cost(inp, cached, outp, cases["ML4 sale"]) - 0.593) < 0.01
assert abs(session_cost(inp, cached, outp, cases["ML4 list"]) - 1.186) < 0.01
print("\nThe sale makes ML4 ~half the session cost of Mistral-hosted GLM-5.3.")

## 3. The verbosity tax: per-token prices lie
Artificial Analysis measured ML4 emitting ~200M tokens on its index vs an 81M median — ~2.5x the output tokens. Effective output price = list price x verbosity.

In [ ]:
verbosity = 200/81
list_output = 4.18          # $/1M output, ML4 list price
effective_output = list_output * verbosity
print(f"Verbosity factor: {verbosity:.2f}x")
print(f"Effective output cost: ${list_output:.2f} -> ${effective_output:.2f}/1M tokens")
# Compare against a terse competitor at median verbosity
k3_effective = 15.00 * 1.0
print(f"ML4 effective output ${effective_output:.2f} vs Kimi K3 ${k3_effective:.2f}")
assert abs(verbosity - 2.469) < 0.01
assert effective_output > list_output
print("\n2.5x verbosity more than erases the output-price edge. Run your own task.")

## 4. Toy: expected cyber score = capability x (1 - refusal rate)
Mistral says closed models score near-zero on reproduce-and-patch because they refuse the task, not because they fail it. This toy shows how a policy score and a capability score collapse into one headline number.

In [ ]:
def expected_score(capability, refusal_rate):
    return capability * (1 - refusal_rate)

# Hypotheticals: same underlying capability, different refusal policies
cap = 0.85
for name, refusal in [("Open ML4-like", 0.03), ("Strict closed A", 0.97), ("Strict closed B", 0.99)]:
    s = expected_score(cap, refusal)
    print(f"{name:16s} capability={cap:.0%} refusal={refusal:.0%} -> headline {100*s:4.1f}%")

ml4_headline, closed_headline = expected_score(cap, 0.03), expected_score(cap, 0.97)
assert ml4_headline / closed_headline > 30, "refusal dominates the headline gap"
print("\nThe 82%-vs-~0% gap is mostly policy, not capability. Mistral's point exactly.")